# Paths

In [1]:
# Set working directories
base.dir =  "/nfs/lab/projects/CellCrosstalk/npod.pancreas/Final/"
assets.dir = paste(base.dir, "Assets/", sep = "") 
results.dir = paste(base.dir, "Results/corrected/", sep = "")
permutation.result.dir = paste(base.dir, "Results/corrected.permutation.test/", sep = "")


# Databases
cc.db = "/nfs/lab/projects/islet_multiomics_stress/analysis/CellChat/Assets/CellChatDB.human.txt"
Gaulton.db.dir = "/nfs/lab/Luca/Assets/LR.Database/GaultonDB_V2.txt"

permutations.test.tmp.dir = paste(permutation.result.dir, "TMP/", sep = "")
permutations.test.res.dir = paste(permutation.result.dir, "Results/", sep = "")

cc.db = "/nfs/lab/projects/islet_multiomics_stress/analysis/CellChat/Assets/CellChatDB.human.txt"

# DBs

# Dataset

In [ ]:
# conditions
condition.ls = c("ND", "Aab", "T1D_early","T1D_late")

colors.conditions = c("ND" = "#6DBE45",
                      "Aab" = "#D1CF55",
                      "T1D_early" = "#D18F55",
                      "T1D_late" = "#D15555")


# Contrasts
Contrast_1 = c(condition.ls[2], condition.ls[3], condition.ls[4])
Contrast_2 = c(condition.ls[1], condition.ls[1], condition.ls[1])

# Celltypes and compartments
Endocrine = c('Alpha', 'Beta', 'Delta')
Exocrine = c('Acinar1_2_6','Acinar_3','Acinar_4','Acinar_5','Ductal','MUC5b_Ductal')
Immune = c('Tcells','Macrophage')
Endothelial = c('Endothelial')
Stromal = c('Activated_Stellate','Quiescent_Stellate')

cell.pop.order = c(Endocrine, Exocrine, Immune, Endothelial, Stromal)
rev.cell.pop.order = c(rev(Stromal), rev(Endothelial), rev(Immune), rev(Exocrine), rev(Endocrine), use.names = TRUE)

compartment = cell.pop.order
compartment = ifelse(compartment  %in%  Endocrine, "Endocrine", 
                 ifelse(compartment  %in%  Exocrine, "Exocrine",
                        ifelse(compartment  %in%  Immune, "Immune",
                               ifelse(compartment  %in%  Endothelial, "Endothelial", "Stromal"))))

Cellpop.compartment = as.data.frame(cbind(cell.pop.order, compartment))
Cellpop.compartment$compartment = factor(Cellpop.compartment$compartment, levels = c('Endocrine','Exocrine','Immune','Endothelial', "Stromal"))
colnames(Cellpop.compartment)[1] = "Celltype"

colors.compartments = c('Endocrine'='#26bfbf',
           'Exocrine'='#ed872d',
           'Immune'='#3F98E0',
           'Endothelial'='#800080',
           'Stromal'='#F5DE6C')

Cellpop.compartment$colors.compartments = compartment
Cellpop.compartment$colors.compartments = mapvalues(Cellpop.compartment$colors.compartments, names(colors.compartments), colors.compartments, warn_missing = TRUE)
gaps.compartments = c(3, 9, 11, 12)

# Ligands

In [ ]:
# Load and prepare Gaulton DB
Gaulton.db.dir = "/nfs/lab/Luca/Assets/LR.Database/GaultonDB_V2.txt"
# Load list of ligands and receptors
SM.ls.df = read.table(paste(assets.dir, "SM.ls.df.txt", sep=""), sep = "\t", stringsAsFactors = F, header = T)
SM.ls = unique(SM.ls.df$gene)
length(SM.ls)

# Load database
cc.db.data = read.table(cc.db, sep = "\t", stringsAsFactors = F, header = T)
# cc.db.data = cc.db.data[,c(11,2,3)]
message("CCDB number of interactions: ", length(levels(factor(cc.db.data$interaction_name))))

gaulton.db.data = read.table(Gaulton.db.dir, sep = "\t", stringsAsFactors = F, header = T)
colnames(gaulton.db.data)[1] = "gene"

SM.ls.df.GDB = merge(SM.ls.df, gaulton.db.data, by = "gene", all.x = TRUE)
SM.ls.df.GDB[is.na(SM.ls.df.GDB)] <- ""

# Calculate category sizes
categories.size = SM.ls.df.GDB[SM.ls.df.GDB$Type == "Ligand",]

# Ligands
ligands.GDB = filter(SM.ls.df.GDB, Type != "Receptor")
colnames(ligands.GDB)[1] = "ligand"
head(ligands.GDB)

# Make a list of all ligands
ligands.ls = c(unique(cc.db.data$ligand), 
              str_split_fixed(unique(cc.db.data$ligand[grepl("_", cc.db.data$ligand)]), "_", n = 2)[,1],
              str_split_fixed(unique(cc.db.data$ligand[grepl("_", cc.db.data$ligand)]), "_", n = 2)[,2])
ligands.ls = unique(ligands.ls)
ligands.ls = ligands.ls[ligands.ls %in% SM.ls]
# Make a list of all receptors
receptors.ls = c(unique(cc.db.data$receptor),
                 str_split_fixed(unique(cc.db.data$receptor[grepl("_", cc.db.data$receptor)]), "_", n = 2)[,1],
                 str_split_fixed(unique(cc.db.data$receptor[grepl("_", cc.db.data$receptor)]), "_", n = 2)[,2])
receptors.ls = unique(receptors.ls)
receptors.ls = receptors.ls[receptors.ls %in% SM.ls]

# Make a list of all receptors that are also ligands
both = ligands.ls[ligands.ls %in% receptors.ls]
message("Number of ligands that are also receptors: ", length(both))


# Exclude those from the other lists
receptors.ls = receptors.ls[!receptors.ls %in% both]
message("Number of receptors: ", length(receptors.ls))


# Exclude those from the other lists
ligands.ls = ligands.ls[!ligands.ls %in% both]
message("Number of ligands: ", length(ligands.ls))


#Check that the length is 860
message("Final Number of SM: ", length(ligands.ls) + length(receptors.ls) + length(both))

Interaction.type.ls = c("Secreted Signaling", "Cell-Cell Contact", "ECM-Receptor")

Functional.classification.order = c('Hormones & Neuropeptides', 'Enzymes', 'Developmental Proteins', 
                                    'Growth Factors and Cytokines', 'Antigen-Presenting Molecules', 'Complement System',
                                    'Immune regulators', 'Modulators', 'Cell Adhesion Molecule', 'ECM Proteins')